In [1]:
!pip install transformers datasets accelerate scikit-learn pandas -q

In [2]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

WINDOW_SIZE = 5  # number of consecutive rows to concatenate
MODEL_NAME = "bert-base-uncased"
BATCH_SIZE = 16
EPOCHS = 10
LR = 2e-5
MAX_LEN = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {DEVICE}")


Using: cuda


In [3]:
df = pd.read_csv("./dataset_match_1_Geo_vs_POR_WCA_2023.csv")

labels = sorted(df['event_class'].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for i, l in enumerate(labels)}

texts = df['text'].tolist()
event_labels = df['event_class'].tolist()
scores = df['highlight_score'].tolist()

X, y_class, y_score = [], [], []
half = WINDOW_SIZE // 2

for i in range(half, len(texts) - half):
    window_text = " ".join(texts[i - half : i + half + 1])
    X.append(window_text)
    y_class.append(label2id[event_labels[i]])
    y_score.append(scores[i])

print(f"Total samples: {len(X)}")
print(f"Score range: {min(y_score):.3f} - {max(y_score):.3f}")


Total samples: 2780
Score range: 0.000 - 1.000


In [4]:
X_train, X_val, yc_train, yc_val, ys_train, ys_val = train_test_split(
    X, y_class, y_score, test_size=0.2, random_state=42, stratify=y_class
)
print(f"Train: {len(X_train)}, Val: {len(X_val)}")


Train: 2224, Val: 556


In [5]:
from transformers import AutoModel

class MultiHeadEventModel(torch.nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size

        # Classification head (event class)
        self.classifier = torch.nn.Sequential(
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden_size, 256),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(256, num_classes)
        )

        # Regression head (excitement score)
        self.regressor = torch.nn.Sequential(
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden_size, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 1),
            torch.nn.Sigmoid()  # score is 0-1
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0, :]  # CLS token

        class_logits = self.classifier(pooled)
        excitement_score = self.regressor(pooled).squeeze(-1)

        return class_logits, excitement_score


In [6]:
class EventDataset(Dataset):
    def __init__(self, texts, class_labels, score_labels, tokenizer, max_len):
        self.texts = texts
        self.class_labels = class_labels
        self.score_labels = score_labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "class_label": torch.tensor(self.class_labels[idx], dtype=torch.long),
            "score_label": torch.tensor(self.score_labels[idx], dtype=torch.float)
        }


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = EventDataset(X_train, yc_train, ys_train, tokenizer, MAX_LEN)
val_dataset = EventDataset(X_val, yc_val, ys_val, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

class_weights = compute_class_weight("balanced", classes=np.arange(len(labels)), y=yc_train)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [8]:
model = MultiHeadEventModel(MODEL_NAME, num_classes=len(labels)).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps)

cls_loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
reg_loss_fn = torch.nn.MSELoss()

# Loss weighting
ALPHA = 0.7  # weight for classification loss
BETA = 0.3   # weight for regression loss


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
best_val_f1 = 0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        class_labels = batch["class_label"].to(DEVICE)
        score_labels = batch["score_label"].to(DEVICE)

        class_logits, pred_scores = model(input_ids, attention_mask)

        loss_cls = cls_loss_fn(class_logits, class_labels)
        loss_reg = reg_loss_fn(pred_scores, score_labels)
        loss = ALPHA * loss_cls + BETA * loss_reg

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()

    # Validation
    model.eval()
    all_preds, all_labels = [], []
    all_pred_scores, all_true_scores = [], []
    val_loss = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            class_labels = batch["class_label"].to(DEVICE)
            score_labels = batch["score_label"].to(DEVICE)

            class_logits, pred_scores = model(input_ids, attention_mask)

            loss_cls = cls_loss_fn(class_logits, class_labels)
            loss_reg = reg_loss_fn(pred_scores, score_labels)
            loss = ALPHA * loss_cls + BETA * loss_reg
            val_loss += loss.item()

            preds = torch.argmax(class_logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(class_labels.cpu().numpy())
            all_pred_scores.extend(pred_scores.cpu().numpy())
            all_true_scores.extend(score_labels.cpu().numpy())

    report = classification_report(all_labels, all_preds, target_names=labels, output_dict=True, zero_division=0)
    val_f1 = report["macro avg"]["f1-score"]

    # Regression metrics
    mae = np.mean(np.abs(np.array(all_pred_scores) - np.array(all_true_scores)))

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f} | F1: {val_f1:.4f} | Score MAE: {mae:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), "best_multihead_model.pt")
        print(f"  → Saved best model")


Epoch 1/10 | Loss: 1.5432 | Val Loss: 1.4720 | F1: 0.1481 | Score MAE: 0.2229
  → Saved best model
Epoch 2/10 | Loss: 1.3484 | Val Loss: 1.0761 | F1: 0.3483 | Score MAE: 0.2197
  → Saved best model
Epoch 3/10 | Loss: 0.8979 | Val Loss: 0.6824 | F1: 0.5361 | Score MAE: 0.2168
  → Saved best model
Epoch 4/10 | Loss: 0.4916 | Val Loss: 0.4673 | F1: 0.7203 | Score MAE: 0.2028
  → Saved best model
Epoch 5/10 | Loss: 0.2423 | Val Loss: 0.3230 | F1: 0.7640 | Score MAE: 0.1856
  → Saved best model
Epoch 6/10 | Loss: 0.1300 | Val Loss: 0.3731 | F1: 0.8146 | Score MAE: 0.1778
  → Saved best model
Epoch 7/10 | Loss: 0.0928 | Val Loss: 0.2849 | F1: 0.8255 | Score MAE: 0.1705
  → Saved best model
Epoch 8/10 | Loss: 0.0595 | Val Loss: 0.3396 | F1: 0.8268 | Score MAE: 0.1653
  → Saved best model
Epoch 9/10 | Loss: 0.0482 | Val Loss: 0.3365 | F1: 0.8369 | Score MAE: 0.1598
  → Saved best model
Epoch 10/10 | Loss: 0.0361 | Val Loss: 0.3819 | F1: 0.8327 | Score MAE: 0.1586


In [11]:
def predict(text_rows, model, tokenizer):
    combined = " ".join(text_rows)
    encoding = tokenizer(combined, truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt")

    # Only pass what our forward() expects
    encoding = {k: v.to(DEVICE) for k, v in encoding.items() if k in ("input_ids", "attention_mask")}

    model.eval()
    with torch.no_grad():
        class_logits, excitement = model(**encoding)
        pred_class = torch.argmax(class_logits, dim=1).item()
        confidence = torch.softmax(class_logits, dim=1)[0][pred_class].item()

    return {
        "event_class": id2label[pred_class],
        "confidence": round(confidence, 3),
        "excitement_score": round(excitement.item(), 4)
    }


In [13]:
def predict_event(text_rows, model, tokenizer):
    combined = " ".join(text_rows)
    encoding = tokenizer(combined, truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt")
    encoding = {k: v.to(DEVICE) for k, v in encoding.items() if k in ("input_ids", "attention_mask")}

    model.eval()
    with torch.no_grad():
        class_logits, excitement = model(**encoding)
        pred_class = torch.argmax(class_logits, dim=1).item()
        confidence = torch.softmax(class_logits, dim=1)[0][pred_class].item()

    return {
        "event_class": id2label[pred_class],
        "confidence": round(confidence, 3),
        "excitement_score": round(excitement.item(), 4)
    }

# Example
sample_rows = texts[50:55]
result = predict_event(sample_rows, model, tokenizer)
print(result)
print(f"Actual: {event_labels[52]}")


{'event_class': 'normal_play', 'confidence': 0.989, 'excitement_score': 0.1166}
Actual: normal_play
